# V04 — Plotly Graph Objects Advanced

**Topics:** Multi-trace figures, secondary y-axes, custom colorbars, 3D charts, contour plots, radar charts, error bars, custom legends.

**Reference:** [Graph Objects docs](https://plotly.com/python/graph-objects/)

**Allowed:** `plotly.graph_objects`, `plotly.express`, `plotly.subplots`, `pandas`, `numpy`


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.datasets import fetch_openml, fetch_california_housing
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)
retail['CustomerID'] = retail['CustomerID'].astype(int)

monthly = retail.groupby('Month').agg(
    Revenue=('Revenue','sum'),
    Orders=('InvoiceNo','nunique'),
    Customers=('CustomerID','nunique'),
    AvgOrderValue=('Revenue', lambda x: x.sum() / retail.loc[x.index,'InvoiceNo'].nunique())
).reset_index()

housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame.copy()
housing.columns = [c.lower() for c in housing.columns]

credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')

print(f"Monthly: {monthly.shape} | Housing: {housing.shape} | Credit: {credit.shape}")

---
## Exercise 1 — Dual Y-Axis Chart

**Spec:** Revenue bars + Customer count line on dual axes.
- Use `make_subplots(specs=[[{'secondary_y': True}]])`
- Primary Y (left): `go.Bar` for Revenue, color `'#1565C0'`, opacity 0.8
- Secondary Y (right): `go.Scatter` for Customers, mode `'lines+markers'`, color `'#E53935'`, line width 2
- Primary Y title: `'Revenue (£)'`, Secondary Y title: `'Unique Customers'`
- Format primary Y as currency: `tickprefix='£'`, `tickformat=',.0f'`
- X: Month
- Title: `'Revenue vs Customer Count by Month'`
- Add legend entries with proper names
- Assign to `fig1`

In [ ]:
# YOUR CODE HERE
fig1 = None
fig1.show()

In [ ]:
# --- ASSERTIONS ---
assert fig1.layout.title.text == 'Revenue vs Customer Count by Month'
trace_types = [t.type for t in fig1.data]
assert 'bar' in trace_types and 'scatter' in trace_types
assert fig1.layout.yaxis.title.text == 'Revenue (£)'
assert fig1.layout.yaxis2.title.text == 'Unique Customers'
assert fig1.layout.yaxis.tickprefix == '£'
print("✓ Exercise 1 passed")

---
## Exercise 2 — Error Bars: Model Confidence

**Spec:** Show mean credit amount by employment duration with 95% confidence interval error bars.

Compute `emp_stats`: group credit by `employment`, compute mean, std, count of `credit_amount`. Compute `ci_95 = 1.96 * std / sqrt(count)`.

- Use `go.Bar` with `error_y=dict(type='data', array=ci_95, visible=True, color='#555', thickness=1.5, width=4)`
- Sort bars by mean credit amount descending
- Color bars by mean value using a colorscale gradient (map mean to color manually using `px.colors.sample_colorscale`)
- Title: `'Mean Credit Amount by Employment Duration (95% CI)'`
- Assign to `fig2`

In [ ]:
emp_stats = (
    credit.groupby('employment')['credit_amount']
    .agg(['mean','std','count']).reset_index()
    .rename(columns={'mean':'mean_credit','std':'std_credit','count':'n'})
)
emp_stats['ci_95'] = 1.96 * emp_stats['std_credit'] / np.sqrt(emp_stats['n'])
emp_stats = emp_stats.sort_values('mean_credit', ascending=False)

# YOUR CODE HERE
fig2 = None
fig2.show()

In [ ]:
# --- ASSERTIONS ---
assert fig2.layout.title.text == 'Mean Credit Amount by Employment Duration (95% CI)'
assert fig2.data[0].error_y.visible == True
assert len(fig2.data[0].error_y.array) == len(emp_stats)
x_vals = list(fig2.data[0].x)
assert x_vals == emp_stats['employment'].tolist(), "Bars must be sorted by mean descending"
print("✓ Exercise 2 passed")

---
## Exercise 3 — Radar / Spider Chart

**Spec:** Compare customer segments on 5 KPI dimensions using a radar chart.

Build `segment_profiles`: 4 customer segments (High Value, Mid Value, Low Value, Churned) each scored 0–100 on: Recency, Frequency, Monetary, Tenure, Engagement. Use synthetic but realistic values.

- Use `go.Scatterpolar` — one trace per segment
- `fill='toself'`
- `opacity=0.6`
- Close the polygon: repeat first category value at end
- Colors: `['#1976D2', '#43A047', '#FB8C00', '#E53935']`
- Title: `'Customer Segment KPI Radar'`
- `polar_radialaxis_range=[0, 100]`
- Assign to `fig3`

In [ ]:
categories = ['Recency', 'Frequency', 'Monetary', 'Tenure', 'Engagement']
segment_profiles = {
    'High Value':  [85, 90, 95, 80, 88],
    'Mid Value':   [60, 65, 60, 55, 62],
    'Low Value':   [35, 30, 25, 40, 32],
    'Churned':     [15, 20, 18, 70, 10],
}
colors = ['#1976D2', '#43A047', '#FB8C00', '#E53935']

# YOUR CODE HERE
fig3 = None
fig3.show()

In [ ]:
# --- ASSERTIONS ---
assert fig3.layout.title.text == 'Customer Segment KPI Radar'
assert len(fig3.data) == 4
for trace in fig3.data:
    assert trace.type == 'scatterpolar'
    assert trace.fill == 'toself'
    # Polygon closed: last theta == first theta
    assert trace.theta[0] == trace.theta[-1], f"Trace {trace.name} not closed"
assert fig3.layout.polar.radialaxis.range == [0, 100]
print("✓ Exercise 3 passed")

**Interpretation:** *(Which segment has the most balanced profile? What does the Churned segment's shape reveal about their behavior before churning?)*

---
## Exercise 4 — 3D Scatter Plot

**Spec:** Visualize 3 housing dimensions simultaneously in 3D space.
- Use `go.Scatter3d`
- X: `medinc`, Y: `houseage`, Z: `medhousval`
- Color: `averooms` — colorscale `'Viridis'`, `showscale=True`
- Marker size: 2, opacity: 0.6
- Use a sample of 3000 rows (`random_state=42`)
- Axis labels: `'Median Income'`, `'House Age'`, `'House Value'`
- Title: `'3D Housing Market Analysis'`
- Set scene camera: `eye=dict(x=1.5, y=1.5, z=0.8)` for a good default angle
- Assign to `fig4`

In [ ]:
housing_3d = housing.sample(3000, random_state=42)

# YOUR CODE HERE
fig4 = None
fig4.show()

In [ ]:
# --- ASSERTIONS ---
assert fig4.layout.title.text == '3D Housing Market Analysis'
assert fig4.data[0].type == 'scatter3d'
assert fig4.data[0].marker.showscale == True
assert fig4.layout.scene.xaxis.title.text == 'Median Income'
assert fig4.layout.scene.zaxis.title.text == 'House Value'
assert fig4.layout.scene.camera.eye.x == 1.5
assert len(fig4.data[0].x) == 3000
print("✓ Exercise 4 passed")

---
## Exercise 5 — Contour Plot: Density Visualization

**Spec:** Show the density of income vs house value using a 2D contour plot.
- Use `go.Histogram2dContour`
- X: `medinc`, Y: `medhousval`
- `colorscale='Hot'`, `reversescale=True`
- `ncontours=20`
- Add a `go.Scatter` overlay of the raw data points: `mode='markers'`, size 2, opacity 0.2, color `'rgba(0,0,0,0.3)'`
- `contours_coloring='heatmap'`
- Title: `'Income vs House Value Density'`
- Assign to `fig5`

In [ ]:
# YOUR CODE HERE
fig5 = None
fig5.show()

In [ ]:
# --- ASSERTIONS ---
assert fig5.layout.title.text == 'Income vs House Value Density'
trace_types = [t.type for t in fig5.data]
assert 'histogram2dcontour' in trace_types
assert 'scatter' in trace_types
contour_trace = [t for t in fig5.data if t.type == 'histogram2dcontour'][0]
assert contour_trace.ncontours == 20
assert contour_trace.reversescale == True
print("✓ Exercise 5 passed")

---
## Exercise 6 — Custom Legend & Multi-Trace Management

**Spec:** Build a chart with a fully customized legend and trace visibility toggles.

Plot monthly Revenue, Orders, Customers, and AvgOrderValue — all normalized to index 100.
- One `go.Scatter` trace per metric
- Colors: `['#1976D2', '#43A047', '#FB8C00', '#8E24AA']`
- Line widths: Revenue=3, others=1.5
- Revenue trace: `visible=True`, others: `visible='legendonly'` (hidden by default, toggleable)
- Custom legend: `orientation='h'`, positioned below chart at `y=-0.2`
- Add `legendgrouptitle_text='Metrics'`
- Title: `'Retail Performance Index (Base 100 = First Month)'`
- Add horizontal reference line at y=100
- Assign to `fig6`

In [ ]:
metrics = ['Revenue', 'Orders', 'Customers', 'AvgOrderValue']
colors = ['#1976D2', '#43A047', '#FB8C00', '#8E24AA']
indexed = monthly.copy()
for m in metrics:
    indexed[m] = monthly[m] / monthly[m].iloc[0] * 100

# YOUR CODE HERE
fig6 = None
fig6.show()

In [ ]:
# --- ASSERTIONS ---
assert fig6.layout.title.text == 'Retail Performance Index (Base 100 = First Month)'
assert len(fig6.data) == 4
revenue_trace = [t for t in fig6.data if t.name == 'Revenue'][0]
assert revenue_trace.visible == True
others = [t for t in fig6.data if t.name != 'Revenue']
assert all(t.visible == 'legendonly' for t in others)
assert fig6.layout.legend.orientation == 'h'
assert fig6.layout.legend.y == -0.2
hlines = [s for s in fig6.layout.shapes if s.type == 'line' and s.y0 == 100]
assert len(hlines) >= 1, "Reference line at 100 missing"
print("✓ Exercise 6 passed")

---
## Exercise 7 — PCA 3D Visualization

**Spec:** Visualize housing clusters in 3D PCA space.

1. Standardize housing features, fit PCA(n_components=3)
2. Create 3 groups using quantile cut on `medhousval` (Low/Mid/High)
3. Use `go.Scatter3d` — one trace per group
4. Marker size 3, opacity 0.6
5. Colors: `['#E53935', '#FB8C00', '#43A047']`
6. Axis labels: `'PC1'`, `'PC2'`, `'PC3'`
7. Add explained variance % to axis labels: e.g. `'PC1 (34.2%)'`
8. Title: `'Housing PCA — 3D Component Space'`
9. Assign to `fig7`

In [ ]:
feature_cols = ['medinc', 'houseage', 'averooms', 'avebedrms', 'population', 'aveoccup']
X_scaled = StandardScaler().fit_transform(housing[feature_cols])
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)
housing_pca = housing.copy()
housing_pca[['PC1','PC2','PC3']] = X_pca
housing_pca['value_group'] = pd.qcut(housing['medhousval'], q=3, labels=['Low','Mid','High'])
evr = pca.explained_variance_ratio_ * 100

# YOUR CODE HERE
fig7 = None
fig7.show()

In [ ]:
# --- ASSERTIONS ---
assert fig7.layout.title.text == 'Housing PCA — 3D Component Space'
assert len(fig7.data) == 3
for trace in fig7.data:
    assert trace.type == 'scatter3d'
pc1_label = fig7.layout.scene.xaxis.title.text
assert 'PC1' in pc1_label and '%' in pc1_label, "PC1 axis must include variance %"
trace_names = {t.name for t in fig7.data}
assert trace_names == {'Low', 'Mid', 'High'}
print("✓ Exercise 7 passed")

---
## Exercise 8 — Animated Heatmap

**Spec:** Animate a monthly revenue heatmap (day-of-week × week-of-month) over months.

Build `daily_retail`: group retail by date, summing Revenue. Add `dow` (0–6), `week_of_month` (1–5), `month`.

Build animation frames — one per month:
- Each frame: pivot `dow` × `week_of_month`, values = Revenue
- Use `go.Figure` with initial `go.Heatmap` trace + `frames` list
- Colorscale: `'YlOrRd'`, fixed `zmin=0`, `zmax=daily_retail['Revenue'].quantile(0.95)`
- Add play/pause buttons via `updatemenus`
- Add slider for month navigation
- Title: `'Daily Revenue Heatmap by Month'`
- Assign to `fig8`

In [ ]:
daily_retail = (
    retail.assign(date=retail['InvoiceDate'].dt.date)
    .groupby('date')['Revenue'].sum().reset_index()
)
daily_retail['date'] = pd.to_datetime(daily_retail['date'])
daily_retail['dow'] = daily_retail['date'].dt.dayofweek
daily_retail['week_of_month'] = daily_retail['date'].apply(
    lambda d: (d.day - 1) // 7 + 1
)
daily_retail['month'] = daily_retail['date'].dt.to_period('M').astype(str)
months = sorted(daily_retail['month'].unique())
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
zmax = daily_retail['Revenue'].quantile(0.95)

# YOUR CODE HERE
fig8 = None
fig8.show()

In [ ]:
# --- ASSERTIONS ---
assert fig8.layout.title.text == 'Daily Revenue Heatmap by Month'
assert fig8.data[0].type == 'heatmap'
assert fig8.frames is not None and len(fig8.frames) == len(months)
assert fig8.data[0].zmax is not None
assert fig8.layout.updatemenus is not None and len(fig8.layout.updatemenus) >= 1
assert fig8.layout.sliders is not None and len(fig8.layout.sliders) >= 1
print("✓ Exercise 8 passed")

---
## Exercise 9 — Parallel Categories: Categorical Flow

**Spec:** Visualize how categorical features relate to credit outcome.
- Use `go.Parcats`
- Dimensions: `['employment', 'purpose', 'housing', 'class']` (in that order)
- Color: `credit['class'].map({'good': 0, 'bad': 1})` — green for good, red for bad
- `colorscale=[[0,'#43A047'],[1,'#E53935']]`
- `line_colorbar_title='Credit Risk'`
- `bundlecolors=True`
- `hoverinfo='count+probability'`
- Title: `'Credit Risk Flow: Employment → Purpose → Housing → Outcome'`
- Assign to `fig9`

In [ ]:
# YOUR CODE HERE
fig9 = None
fig9.show()

In [ ]:
# --- ASSERTIONS ---
assert fig9.layout.title.text == 'Credit Risk Flow: Employment → Purpose → Housing → Outcome'
assert fig9.data[0].type == 'parcats'
dim_labels = [d.label for d in fig9.data[0].dimensions]
assert dim_labels == ['employment', 'purpose', 'housing', 'class']
assert fig9.data[0].bundlecolors == True
print("✓ Exercise 9 passed")

---
## Exercise 10 — Capstone: Dynamic Chart Factory

**Spec:** Build a `ChartFactory` class that generates standardized charts for any business metric.

Implement:
```python
class ChartFactory:
    def trend_with_forecast(df, x, y, forecast_periods=3) -> go.Figure
    def distribution_with_stats(df, col, group_col=None) -> go.Figure
    def ranked_comparison(df, category_col, value_col, top_n=10) -> go.Figure
```

Rules:
- `trend_with_forecast`: line chart + simple linear extrapolation shown as dashed line with CI shading for `forecast_periods` additional points
- `distribution_with_stats`: histogram + KDE approximation (use numpy histogram + gaussian smoothing) + vertical lines for mean/median
- `ranked_comparison`: horizontal bar chart, sorted, colored by value, with value labels
- Each method applies the corporate theme from V03 Exercise 10
- Each returns a `go.Figure`

In [ ]:
class ChartFactory:

    @staticmethod
    def trend_with_forecast(df: pd.DataFrame, x: str, y: str,
                             forecast_periods: int = 3) -> go.Figure:
        # YOUR CODE HERE
        pass

    @staticmethod
    def distribution_with_stats(df: pd.DataFrame, col: str,
                                  group_col: str = None) -> go.Figure:
        # YOUR CODE HERE
        pass

    @staticmethod
    def ranked_comparison(df: pd.DataFrame, category_col: str,
                           value_col: str, top_n: int = 10) -> go.Figure:
        # YOUR CODE HERE
        pass

# Test all 3
fig_trend = ChartFactory.trend_with_forecast(monthly, 'Month', 'Revenue', forecast_periods=3)
fig_dist = ChartFactory.distribution_with_stats(credit, 'credit_amount', group_col='class')
fig_rank = ChartFactory.ranked_comparison(
    retail.groupby('Country')['Revenue'].sum().reset_index(), 'Country', 'Revenue', top_n=10
)
fig_trend.show()
fig_dist.show()
fig_rank.show()

In [ ]:
# --- ASSERTIONS ---
import plotly.basedatatypes
for fig, name in [(fig_trend,'trend'), (fig_dist,'dist'), (fig_rank,'rank')]:
    assert isinstance(fig, plotly.basedatatypes.BaseFigure), f"{name} must return go.Figure"
    assert fig.layout.paper_bgcolor == '#FAFAFA', f"{name} must apply corporate theme"

# Trend: must have actual + forecast traces
trace_names = [t.name for t in fig_trend.data]
assert any('forecast' in n.lower() or 'Forecast' in n for n in trace_names), "Must have forecast trace"

# Distribution: must have mean/median lines
shapes = fig_dist.layout.shapes
assert len(shapes) >= 2, "Must have mean and median lines"

# Ranked: top_n bars
assert len(fig_rank.data[0].y) <= 10
print("✓ Exercise 10 passed")